=============================================================================
STUDENT 2 — DOCUMENT ANALYST (BUILT FROM SCRATCH)
Multimodal Crime / Incident Report Analyzer
AI for Engineers — Group Assignment
=============================================================================
DATASET: Arkansas Police Department 1033 Training Plan Proposals
HOW TO USE:
1. Download PDF from muckrock.com (link in assignment)
2. Upload to Google Drive:
MyDrive/multimodal_crime_analyzer/pdf_data/LESO2 (1).pdf
3. Upload THIS notebook to Google Colab
4. Run All cells
=============================================================================

### Mount Google Drive

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


### Install All Dependencies

In [2]:
import subprocess
subprocess.run(["pip", "install", "pdfplumber", "pymupdf", "spacy",
                "pytesseract", "pandas", "pillow", "-q"])
subprocess.run(["python", "-m", "spacy", "download", "en_core_web_sm", "-q"])
subprocess.run(["apt-get", "install", "-y", "tesseract-ocr", "-q"])
print("✔ All dependencies installed")

✔ All dependencies installed


### Imports

In [3]:
import os
import re
import warnings
import pandas as pd
import pdfplumber
import fitz
import spacy
import pytesseract
from PIL import Image
from IPython.display import display

warnings.filterwarnings("ignore")
print("✔ Imports done")

✔ Imports done


### Configuration

In [4]:
PDF_PATH   = "/content/drive/MyDrive/AI_DATASETS/DOCUMENT(PDF)_AI/LESO2 (1).pdf"
OUTPUT_CSV = "/content/drive/MyDrive/AI_DATASETS/pdf_output.csv"

# Minimum characters for a page to be considered meaningful
MIN_PAGE_CHARS = 100

### Load spaCy NLP Model

In [5]:
print("Loading spaCy NER model...")
nlp = spacy.load("en_core_web_sm")
print("✔ spaCy loaded\n")

# =============================================================================
#  PART 1 — RAW TEXT EXTRACTION
#  Following assignment tip: pdfplumber extracts text-based PDFs cleanly
#  OCR fallback handles any scanned pages automatically
# =============================================================================

Loading spaCy NER model...
✔ spaCy loaded



### Extract Text Page by Page

In [6]:
print("=" * 55)
print("  PART 1 — PDF TEXT EXTRACTION (page by page)")
print("=" * 55)

if not os.path.exists(PDF_PATH):
    print(f"❌ PDF not found: {PDF_PATH}")
    print("   Check your path in CELL 4")
    page_texts = []
else:
    page_texts = []

    # Open fitz once outside the loop — fixes resource leak loophole
    fitz_doc = fitz.open(PDF_PATH)

    with pdfplumber.open(PDF_PATH) as pdf:
        total_pages = len(pdf.pages)
        print(f"  Total pages: {total_pages}\n")

        for i, page in enumerate(pdf.pages):
            # Try pdfplumber first (text-based PDF)
            text = page.extract_text() or ""

            # Fallback to OCR if text too short (scanned page)
            if len(text.strip()) < MIN_PAGE_CHARS:
                try:
                    pix = fitz_doc[i].get_pixmap(dpi=200)
                    img = Image.frombytes("RGB", [pix.width, pix.height], pix.samples)
                    text = pytesseract.image_to_string(img)
                except Exception:
                    text = ""

            page_texts.append({
                "page_num": i + 1,
                "text":     text.strip()
            })

    fitz_doc.close()  # Always close fitz properly
    print(f"✔ PART 1 COMPLETE — {total_pages} pages extracted")

# =============================================================================
#  PART 2 — STRUCTURED INFORMATION EXTRACTION
# =============================================================================

  PART 1 — PDF TEXT EXTRACTION (page by page)
  Total pages: 75

✔ PART 1 COMPLETE — 75 pages extracted


### Define Extraction Maps

In [7]:
print("\n" + "=" * 55)
print("  PART 2 — STRUCTURED INFO EXTRACTION")
print("=" * 55)

# LOOPHOLE FIX: More specific date pattern — avoids matching page numbers
# like "Page 1 of 2" or "Number: 411.3"
DATE_PATTERN = re.compile(
    r"\b("
    r"\d{1,2}/\d{1,2}/\d{4}"               # MM/DD/YYYY
    r"|\d{1,2}-\d{1,2}-\d{4}"              # MM-DD-YYYY
    r"|(?:Jan(?:uary)?|Feb(?:ruary)?|Mar(?:ch)?|Apr(?:il)?|May|Jun(?:e)?"
    r"|Jul(?:y)?|Aug(?:ust)?|Sep(?:tember)?|Oct(?:ober)?|Nov(?:ember)?"
    r"|Dec(?:ember)?)\s+\d{1,2},?\s+\d{4}" # Month DD, YYYY
    r"|\d{1,2}\s+(?:Jan(?:uary)?|Feb(?:ruary)?|Mar(?:ch)?|Apr(?:il)?|May"
    r"|Jun(?:e)?|Jul(?:y)?|Aug(?:ust)?|Sep(?:tember)?|Oct(?:ober)?"
    r"|Nov(?:ember)?|Dec(?:ember)?)\s+\d{4}" # DD Month YYYY
    r")\b",
    re.IGNORECASE
)

# LOOPHOLE FIX: More specific doc type — check longer keywords first
# to avoid "1033" matching both Training Proposal and Law Enforcement
DOC_TYPE_MAP = [
    ("Standard Operating Procedure", ["standard operating procedure", "policies and procedures", "sop"]),
    ("Memorandum",                   ["memorandum"]),
    ("Invoice",                      ["invoice"]),
    ("Certificate",                  ["certificate of completion", "this certificate is awarded"]),
    ("Aircraft Request",             ["aircraft request", "lea aircraft"]),
    ("Lesson Plan",                  ["lesson plan", "lesson title", "learning goal"]),
    ("1033 Training Proposal",       ["1033", "training plan", "training proposal", "leso"]),
    ("Incident Report",              ["incident report", "offense report"]),
]

# LOOPHOLE FIX: Program priority order — most specific first
# MRAP is checked before Law Enforcement since many pages mention both
PROGRAM_MAP = [
    ("Aviation Unit",            ["aviation unit", "helicopter", "aircraft request"]),
    ("MRAP Operations",          ["mrap", "mine resistant ambush", "armored rescue vehicle"]),
    ("SWAT / Tactical",          ["swat", "special weapons", "special response team", "sort"]),
    ("Narcotics Division",       ["narcotic", "drug enforcement", "dea"]),
    ("Community Policing",       ["community policing"]),
    ("Law Enforcement Support",  ["law enforcement support", "leso", "1033 program"]),
]

KEY_DETAIL_WORDS = ["equipment", "request", "training", "vehicle", "mrap",
                     "obtained", "tactical", "policy", "purpose", "allocated",
                     "assigned", "acquired", "implemented"]

# Noise pages to skip — certificate/signature pages with little info
NOISE_PATTERNS = ["this certificate is awarded to", "joint program office",
                   "smith integrated technologies", "jpо mrap apm"]

print("✔ Extraction maps defined")


  PART 2 — STRUCTURED INFO EXTRACTION
✔ Extraction maps defined


### Define Extraction Functions

In [8]:
def clean_text(text):
    """Normalize whitespace and remove OCR artifacts."""
    text = re.sub(r'\s+', ' ', text)          # Multiple spaces → single
    text = re.sub(r'[^\x20-\x7E\n]', ' ', text)  # Remove non-ASCII chars
    return text.strip()


def is_noise_page(text):
    """
    LOOPHOLE FIX: Skip certificate pages, blank pages, and image-only pages
    that have no structured information worth extracting.
    """
    text_lower = text.lower()
    # Too short
    if len(text_lower) < MIN_PAGE_CHARS:
        return True
    # Certificate / noise patterns
    if any(p in text_lower for p in NOISE_PATTERNS):
        return True
    # Pages that are mostly numbers/symbols (forms with checkboxes)
    alpha_ratio = sum(c.isalpha() for c in text) / max(len(text), 1)
    if alpha_ratio < 0.3:
        return True
    return False


def extract_department(text):
    """
    LOOPHOLE FIX: Multi-strategy department extraction.
    1. Look for known department patterns in first 10 lines
    2. spaCy NER as fallback
    3. Normalize variations (Sheriff vs Sheriff's Office)
    """
    lines = [l.strip() for l in text.split("\n") if len(l.strip()) > 5]

    # Strategy 1: Header lines (first 10 lines usually have dept name)
    dept_keywords = ["police department", "sheriff's office", "sheriff's department",
                     "sheriff department", "police dept", "sheriff office"]
    for line in lines[:10]:
        line_lower = line.lower()
        if any(kw in line_lower for kw in dept_keywords):
            # Clean up the line
            dept = re.sub(r'\s+', ' ', line).strip()
            # Remove common prefixes
            dept = re.sub(r'^(to:|from:|re:|subject:)\s*', '', dept, flags=re.IGNORECASE)
            if len(dept) > 5:
                return dept

    # Strategy 2: spaCy NER
    doc  = nlp(text[:3000])
    orgs = [e.text.strip() for e in doc.ents if e.label_ == "ORG"]
    for org in orgs:
        if any(kw in org.lower() for kw in ["police", "sheriff", "department", "office"]):
            return re.sub(r'\s+', ' ', org).strip()

    # Strategy 3: From/Sender line
    for line in lines[:15]:
        if line.lower().startswith("from:"):
            return line[5:].strip()

    return "Unknown"


def extract_doc_type(text):
    """
    LOOPHOLE FIX: Use ordered list so more specific types checked first.
    Prevents '1033' from matching before 'Standard Operating Procedure'.
    """
    text_lower = text.lower()
    for doc_type, keywords in DOC_TYPE_MAP:
        if any(kw in text_lower for kw in keywords):
            return doc_type
    return "Police Document"


def extract_date(text):
    """
    LOOPHOLE FIX: Filter out false dates like page numbers.
    Only accept dates where surrounding context looks like a real date field.
    """
    matches = DATE_PATTERN.findall(text)
    for date_str in matches:
        date_str = date_str.strip()
        # Skip very short matches that might be page numbers
        if len(date_str) < 6:
            continue
        # Try to parse to standard format
        try:
            from datetime import datetime
            for fmt in ["%m/%d/%Y", "%m-%d-%Y", "%B %d, %Y",
                        "%b %d, %Y", "%d %B %Y", "%d %b %Y",
                        "%B %d %Y", "%b %d %Y"]:
                try:
                    return datetime.strptime(date_str, fmt).strftime("%Y-%m-%d")
                except ValueError:
                    continue
        except Exception:
            pass
        return date_str
    return "Unknown"


def extract_program(text):
    """
    LOOPHOLE FIX: Use ordered list — most specific program checked first.
    Aviation Unit checked before Law Enforcement so LRPD aviation pages
    don't get tagged as Law Enforcement Support.
    """
    text_lower = text.lower()
    for program, keywords in PROGRAM_MAP:
        if any(kw in text_lower for kw in keywords):
            return program
    return "General Law Enforcement"


def extract_key_detail(text):
    """
    LOOPHOLE FIX: Skip very short or boilerplate sentences.
    Return the most informative sentence — not just the first one.
    """
    sentences = [s.strip() for s in re.split(r'[.\n]', text) if len(s.strip()) > 40]
    # Score sentences by number of key words
    best_sent  = None
    best_score = 0
    for sent in sentences[:20]:  # Check first 20 sentences
        score = sum(1 for kw in KEY_DETAIL_WORDS if kw in sent.lower())
        if score > best_score:
            best_score = score
            best_sent  = sent
    if best_sent:
        clean = " ".join(best_sent.split())
        return (clean[:120] + "...") if len(clean) > 120 else clean
    return (sentences[0][:120] + "...") if sentences else "N/A"


print("✔ All extraction functions defined")

✔ All extraction functions defined


### Process Each Page

In [9]:
print("\nProcessing each page...\n")

records = []

for page in page_texts:
    text    = clean_text(page["text"])
    page_no = page["page_num"]

    # LOOPHOLE FIX: Skip noise pages (certificates, blank, image-only)
    if is_noise_page(text):
        continue

    department = extract_department(text)
    doc_type   = extract_doc_type(text)
    date       = extract_date(text)
    program    = extract_program(text)
    key_detail = extract_key_detail(text)

    # Only keep pages with at least department or date found
    if department != "Unknown" or date != "Unknown":
        records.append({
            "page_num":   page_no,
            "Department": department,
            "Doc_Type":   doc_type,
            "Date":       date,
            "Program":    program,
            "Key_Detail": key_detail,
        })

print(f"  Found {len(records)} meaningful pages")


Processing each page...

  Found 43 meaningful pages


### Smart Grouping — Department + Program

In [10]:
df_all = pd.DataFrame(records)

# LOOPHOLE FIX: Clean department names properly
# Remove newlines, extra spaces, and common OCR artifacts
df_all["Department"] = (
    df_all["Department"]
    .str.replace(r'\n.*', '', regex=True)     # Remove everything after newline
    .str.replace(r'\s+', ' ', regex=True)     # Normalize spaces
    .str.strip()
)

# LOOPHOLE FIX: Group by Department AND Program
# Same dept with different programs → separate rows
# Same dept with same program → keep first (most complete page)
df_grouped = (
    df_all
    .groupby(["Department", "Program"], as_index=False)
    .first()
)

# Remove Unknown departments if real ones exist
real = df_grouped[df_grouped["Department"] != "Unknown"]
if len(real) > 0:
    df_grouped = real.copy()

# Sort by page number to match PDF order
if "page_num" in df_grouped.columns:
    df_grouped = df_grouped.sort_values("page_num").reset_index(drop=True)

# LOOPHOLE FIX: Re-number Report IDs cleanly after all filtering
df_grouped["Report_ID"] = [f"RPT_{i+1:03d}" for i in range(len(df_grouped))]

print(f"\n✔ {len(df_grouped)} records after smart grouping\n")
for _, row in df_grouped.iterrows():
    print(f"  ✔ {row['Report_ID']}: {row['Department'][:40]} | "
          f"{row['Program']} | {row['Date']}")


✔ 42 records after smart grouping

  ✔ RPT_001: Sheriff Kelley Cradduck Major Nathan Atc | MRAP Operations | 2015-05-26
  ✔ RPT_002: Benton County Sheriff s Office POLICIES  | MRAP Operations | Unknown
  ✔ RPT_003: vi. Monthly inspection, to include runni | MRAP Operations | Unknown
  ✔ RPT_004: Il. III. ARMORED RESCUE VEHICLE (ARV) OP | MRAP Operations | Unknown
  ✔ RPT_005: Benton Police Department Policy: Effecti | MRAP Operations | 2015-04-24
  ✔ RPT_006: STATE OF ARKANSAS Commission on Law Enfo | MRAP Operations | 2014-07-01
  ✔ RPT_007: Bryant Police Department 312 Roya Lane B | MRAP Operations | 2014-07-02
  ✔ RPT_008: CABOT POLICE DEPARTMENT CABOT, ARKANSAS  | MRAP Operations | Unknown
  ✔ RPT_009: G. Fire Drill t {O00 SECON ORO) IS Crew  | MRAP Operations | Unknown
  ✔ RPT_010: Cabot Police Department Road Course and  | MRAP Operations | Unknown
  ✔ RPT_011: /2403-1 352 Sheriff Marty Boyd mboyd@ ti | MRAP Operations | 2015-05-29
  ✔ RPT_012: e Falls, smashed and pinched extre

### Save and Display Output

In [11]:
# Assignment output — exact columns
assignment_df = df_grouped[["Report_ID", "Department", "Doc_Type",
                              "Date", "Program", "Key_Detail"]].copy()

# Save full output (with page numbers and program)
df_grouped.to_csv(OUTPUT_CSV.replace(".csv", "_full.csv"), index=False)

# Save assignment output
assignment_df.to_csv(OUTPUT_CSV, index=False)

print(f"\n✔ pdf_output.csv saved      — {len(assignment_df)} records (submit this)")
print(f"✔ pdf_output_full.csv saved — includes page numbers")
print(f"\n{'='*55}")
print("  STRUCTURED OUTPUT TABLE")
print(f"{'='*55}")
display(assignment_df)


✔ pdf_output.csv saved      — 42 records (submit this)
✔ pdf_output_full.csv saved — includes page numbers

  STRUCTURED OUTPUT TABLE


,Report_ID,Department,Doc_Type,Date,Program,Key_Detail
0,RPT_001,Sheriff Kelley Cradduck Major Nathan Atchison ...,1033 Training Proposal,2015-05-26,MRAP Operations,"As of that date, our office has implemented a ..."
1,RPT_002,Benton County Sheriff s Office POLICIES AND PR...,Standard Operating Procedure,Unknown,MRAP Operations,3 6/1/15 6/1/16 ISSUE DATE | 6/1/15 REVISION T...
2,RPT_003,"vi. Monthly inspection, to include running the...",Police Document,Unknown,MRAP Operations,"TRAINING AND OPERATION: I Zi, No personnel wil..."
3,RPT_004,Il. III. ARMORED RESCUE VEHICLE (ARV) OPERATIO...,Lesson Plan,Unknown,MRAP Operations,ARMORED RESCUE VEHICLE (ARV) OPERATIONS Lesson...
4,RPT_005,Benton Police Department Policy: Effective: 4/...,Police Document,2015-04-24,MRAP Operations,Benton Police Department Policy: Effective: 4/...
5,RPT_006,STATE OF ARKANSAS Commission on Law Enforcemen...,Police Document,2014-07-01,MRAP Operations,"Calnan, In response to your applications(s) fo..."
6,RPT_007,"Bryant Police Department 312 Roya Lane Bryant,...",1033 Training Proposal,2014-07-02,MRAP Operations,It is the policy of the Bryant Police Departme...
7,RPT_008,"CABOT POLICE DEPARTMENT CABOT, ARKANSAS 72023 ...",Police Document,Unknown,MRAP Operations,Purpose: Mine Resistant Ambush Protected Vehic...
8,RPT_009,G. Fire Drill t {O00 SECON ORO) IS Crew announ...,Police Document,Unknown,MRAP Operations,All documented training on the MRAP will be fo...
9,RPT_010,Cabot Police Department Road Course and Traini...,Police Document,Unknown,MRAP Operations,Cabot Police Department Road Course and Traini...


### Verify Output Columns

In [12]:
print(f"\n{'='*55}")
print("  OUTPUT COLUMNS CHECK")
print(f"{'='*55}")
expected = ["Report_ID", "Department", "Doc_Type", "Date", "Program", "Key_Detail"]
for col in expected:
    status = "✔" if col in assignment_df.columns else "❌"
    print(f"  {status} {col}")

print(f"\n{'='*55}")
print("  PROGRAM DISTRIBUTION")
print(f"{'='*55}")
display(assignment_df["Program"].value_counts().reset_index().rename(
    columns={"index": "Program", "Program": "Count"}
))

print(f"\n{'='*55}")
print("  DOC TYPE DISTRIBUTION")
print(f"{'='*55}")
display(assignment_df["Doc_Type"].value_counts().reset_index().rename(
    columns={"index": "Doc_Type", "Doc_Type": "Count"}
))

print(f"\n✔ Download pdf_output.csv from your Google Drive")
print(f"  Share with your team for the integration step.")


  OUTPUT COLUMNS CHECK
  ✔ Report_ID
  ✔ Department
  ✔ Doc_Type
  ✔ Date
  ✔ Program
  ✔ Key_Detail

  PROGRAM DISTRIBUTION


,Count,count
0,MRAP Operations,26
1,Aviation Unit,12
2,General Law Enforcement,4



  DOC TYPE DISTRIBUTION


,Count,count
0,Police Document,20
1,1033 Training Proposal,13
2,Standard Operating Procedure,3
3,Lesson Plan,3
4,Aircraft Request,2
5,Memorandum,1



✔ Download pdf_output.csv from your Google Drive
  Share with your team for the integration step.
